In [2]:
"""ROUTINA — Daily Habit Tracker UI

Tkinter frontend + a small SQLite database. Core application/business logic can
be connected in handle_ui_action().

JPG asset placeholders (optional):
    assets/background.jpg  -> background used on all screens
    assets/button.jpg      -> image used for menu/action buttons

The app still runs if those files or Pillow are absent; it uses a black/green
fallback theme. If running from Jupyter, the database and assets are resolved
from the current working directory. When run as a .py file, they are resolved
next to this file.
"""

from __future__ import annotations

import sqlite3
import sys
from datetime import date, datetime
from pathlib import Path
import tkinter as tk
from tkinter import messagebox, simpledialog

APP_NAME = "ROUTINA — Daily Habit Tracker"
GREEN = "#20d20d"
BLACK = "#000000"
WHITE = "#f7f7f7"
DARK = "#101010"
MUTED = "#a9a9a9"

# Placeholder asset paths: replace these JPG files with your own assets.
# JPG loading needs Pillow (`pip install pillow`). Missing assets are harmless.
if "__file__" in globals():
    APP_DIR = Path(__file__).resolve().parent
else:
    APP_DIR = Path.cwd()
ASSET_DIR = APP_DIR / "assets"
BACKGROUND_JPG = ASSET_DIR / "background.jpg"  # PLACEHOLDER asset path
BUTTON_JPG = ASSET_DIR / "button.jpg"            # PLACEHOLDER asset path
DB_PATH = APP_DIR / "routina_data.sqlite3"

try:
    from PIL import Image, ImageTk
    PIL_AVAILABLE = True
except ImportError:
    PIL_AVAILABLE = False


class RoutinaApp:
    def __init__(self, root: tk.Tk):
        self.root = root
        self.root.title(APP_NAME)
        self.root.geometry("900x720")
        self.root.minsize(620, 540)
        self.root.configure(bg=BLACK)
        self.db = sqlite3.connect(DB_PATH)
        self.db.row_factory = sqlite3.Row
        self._create_database()
        self._image_refs = []
        self._background_label = None
        self._button_image = None
        self._load_assets()
        self.show_home()
        self.root.protocol("WM_DELETE_WINDOW", self.quit_app)

    def _create_database(self):
        """Create the external database tables if they do not already exist."""
        with self.db:
            self.db.execute("""CREATE TABLE IF NOT EXISTS habits (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT NOT NULL,
                schedule TEXT NOT NULL CHECK(schedule IN ('Every day','Weekdays','Weekends')),
                created_at TEXT NOT NULL
            )""")
            self.db.execute("""CREATE TABLE IF NOT EXISTS habit_completions (
                habit_id INTEGER NOT NULL,
                completion_date TEXT NOT NULL,
                completed INTEGER NOT NULL DEFAULT 1,
                PRIMARY KEY (habit_id, completion_date),
                FOREIGN KEY (habit_id) REFERENCES habits(id) ON DELETE CASCADE
            )""")
            self.db.execute("""CREATE TABLE IF NOT EXISTS tasks (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT NOT NULL,
                completed INTEGER NOT NULL DEFAULT 0,
                created_at TEXT NOT NULL
            )""")

    def _load_assets(self):
        if not PIL_AVAILABLE:
            return
        try:
            if BACKGROUND_JPG.exists():
                img = Image.open(BACKGROUND_JPG).convert("RGB")
                img.thumbnail((1400, 1000))
                self._background_image = ImageTk.PhotoImage(img)
            if BUTTON_JPG.exists():
                img = Image.open(BUTTON_JPG).convert("RGB")
                self._button_image = ImageTk.PhotoImage(img.resize((280, 62)))
        except Exception as exc:
            # Placeholder fallback: if an image is invalid, use themed Tk widgets.
            print(f"Could not load optional JPG assets: {exc}")

    def _clear(self):
        for child in self.root.winfo_children():
            child.destroy()
        self._image_refs = []
        if PIL_AVAILABLE and hasattr(self, "_background_image"):
            self._background_label = tk.Label(self.root, image=self._background_image, bd=0)
            self._background_label.place(relx=0, rely=0, relwidth=1, relheight=1)
            self._background_label.lower()

    def _button(self, parent, text, command, *, width=22, font_size=18, accent=True):
        # Uses the optional JPG button asset when provided; otherwise styled Tk button.
        if self._button_image is not None:
            btn = tk.Button(parent, text=text, image=self._button_image, compound="center",
                            command=command, fg=GREEN if accent else WHITE,
                            bg=BLACK, activebackground="#202020", activeforeground=GREEN,
                            font=("Arial", font_size, "italic"), bd=0, cursor="hand2")
        else:
            btn = tk.Button(parent, text=text, command=command, width=width,
                            fg=GREEN if accent else WHITE, bg=BLACK,
                            activebackground="#202020", activeforeground=GREEN,
                            font=("Arial", font_size, "italic"), bd=3,
                            relief="ridge", highlightthickness=0, cursor="hand2",
                            padx=16, pady=10)
        return btn

    def _title(self, parent, title, subtitle=None):
        tk.Label(parent, text=title, bg=BLACK, fg=GREEN,
                 font=("Arial", 34, "bold italic")).pack(pady=(18, 0))
        if subtitle:
            tk.Label(parent, text=subtitle, bg=BLACK, fg=GREEN,
                     font=("Arial", 14, "italic")).pack(pady=(0, 16))

    def handle_ui_action(self, action, data=None):
        """Integration placeholder for your own core Python code.

        Replace the print statement with your own calls, for example:
            if action == "open_habits":
                your_core_function()
        The UI and database already work; no external core logic is required.
        """
        print(f"UI ACTION: {action} | data={data}")  # PLACEHOLDER hook for Line A

    def show_home(self):
        self._clear()
        frame = tk.Frame(self.root, bg=BLACK)
        frame.place(relx=0.5, rely=0.48, anchor="center")
        self._title(frame, "ROUTINA", "Daily Habit Tracker")
        tk.Frame(frame, bg=BLACK, height=90).pack()
        self._button(frame, "Habits", self._open_habits, width=24, font_size=25).pack(pady=10, ipady=7)
        self._button(frame, "Tasks", self._open_tasks, width=20, font_size=25).pack(pady=10, ipady=7)
        self._button(frame, "Quit", self.quit_app, width=20, font_size=25).pack(pady=10, ipady=7)

    def _open_habits(self):
        self.handle_ui_action("open_habits")
        self.show_habits()

    def _open_tasks(self):
        self.handle_ui_action("open_tasks")
        self.show_tasks()

    def show_habits(self):
        self._clear()
        top = tk.Frame(self.root, bg=BLACK)
        top.pack(fill="x", padx=18)
        self._button(top, "← Home", self.show_home, width=10, font_size=12, accent=False).pack(side="left", pady=12)
        self._title(self.root, "Habits", "Tap a habit to toggle today's completion")

        controls = tk.Frame(self.root, bg=BLACK)
        controls.pack(pady=(0, 8))
        self._button(controls, "+ Add habit", self.add_habit, width=13, font_size=12).pack(side="left", padx=5)
        self._button(controls, "Remove habit", self.remove_habit, width=13, font_size=12, accent=False).pack(side="left", padx=5)

        canvas = tk.Canvas(self.root, bg=BLACK, highlightthickness=0)
        scrollbar = tk.Scrollbar(self.root, orient="vertical", command=canvas.yview)
        body = tk.Frame(canvas, bg=BLACK)
        body.bind("<Configure>", lambda _e: canvas.configure(scrollregion=canvas.bbox("all")))
        canvas.create_window((0, 0), window=body, anchor="nw")
        canvas.configure(yscrollcommand=scrollbar.set)
        canvas.pack(side="left", fill="both", expand=True, padx=(22, 0), pady=8)
        scrollbar.pack(side="right", fill="y", padx=(0, 12), pady=8)

        today = date.today()
        section = "Weekends" if today.weekday() >= 5 else "Weekdays"
        # Every day habits appear in either day group, so daily habits are always visible.
        self._habit_section(body, "WEEKDAYS", ("Weekdays", "Every day"))
        self._habit_section(body, "WEEKENDS", ("Weekends", "Every day"))

    def _habit_section(self, parent, heading, schedules):
        section = tk.Frame(parent, bg=BLACK, highlightbackground="#454545", highlightthickness=1)
        section.pack(fill="x", expand=True, padx=8, pady=7)
        tk.Label(section, text=heading, bg=BLACK, fg=GREEN,
                 font=("Arial", 16, "bold")).pack(anchor="w", padx=12, pady=(8, 3))
        placeholders = ",".join("?" for _ in schedules)
        rows = self.db.execute(
            f"SELECT * FROM habits WHERE schedule IN ({placeholders}) ORDER BY id", schedules
        ).fetchall()
        if not rows:
            tk.Label(section, text="No habits yet — use + Add habit.", bg=BLACK,
                     fg=MUTED, font=("Arial", 11)).pack(anchor="w", padx=14, pady=8)
        for habit in rows:
            done_row = self.db.execute(
                "SELECT completed FROM habit_completions WHERE habit_id=? AND completion_date=?",
                (habit["id"], date.today().isoformat())
            ).fetchone()
            done = bool(done_row and done_row["completed"])
            label = ("✓  " if done else "○  ") + habit["name"]
            btn = self._button(section, label,
                               lambda hid=habit["id"]: self.toggle_habit(hid),
                               width=35, font_size=13, accent=not done)
            btn.pack(fill="x", padx=12, pady=4)
        tk.Frame(section, bg=BLACK, height=5).pack()

    def add_habit(self):
        name = simpledialog.askstring("Add habit", "Enter a name for this habit:", parent=self.root)
        if not name or not name.strip():
            return
        schedule = self._schedule_dialog()
        if schedule is None:
            return
        with self.db:
            self.db.execute("INSERT INTO habits(name, schedule, created_at) VALUES(?,?,?)",
                            (name.strip(), schedule, datetime.now().isoformat(timespec="seconds")))
        self.handle_ui_action("add_habit", {"name": name.strip(), "schedule": schedule})
        self.show_habits()

    def _schedule_dialog(self):
        win = tk.Toplevel(self.root)
        win.title("Habit schedule")
        win.configure(bg=BLACK)
        win.transient(self.root)
        win.grab_set()
        tk.Label(win, text="Repeat this habit on:", bg=BLACK, fg=WHITE,
                 font=("Arial", 12)).pack(padx=22, pady=(18, 8))
        choice = tk.StringVar(value="Every day")
        for option in ("Every day", "Weekdays", "Weekends"):
            tk.Radiobutton(win, text=option, variable=choice, value=option,
                           bg=BLACK, fg=GREEN, selectcolor=DARK, activebackground=BLACK,
                           activeforeground=GREEN).pack(anchor="w", padx=25)
        result = {"value": None}
        def save():
            result["value"] = choice.get()
            win.destroy()
        def cancel():
            win.destroy()
        self._button(win, "Save", save, width=12, font_size=12).pack(pady=(12, 5))
        self._button(win, "Cancel", cancel, width=12, font_size=12, accent=False).pack(pady=(0, 14))
        self.root.wait_window(win)
        return result["value"]

    def toggle_habit(self, habit_id):
        today = date.today().isoformat()
        row = self.db.execute(
            "SELECT completed FROM habit_completions WHERE habit_id=? AND completion_date=?",
            (habit_id, today)
        ).fetchone()
        new_value = 0 if row and row["completed"] else 1
        with self.db:
            self.db.execute("""INSERT INTO habit_completions(habit_id, completion_date, completed)
                VALUES(?,?,?) ON CONFLICT(habit_id, completion_date)
                DO UPDATE SET completed=excluded.completed""", (habit_id, today, new_value))
        self.handle_ui_action("toggle_habit", {"habit_id": habit_id, "completed": bool(new_value)})
        self.show_habits()

    def remove_habit(self):
        rows = self.db.execute("SELECT id, name, schedule FROM habits ORDER BY id").fetchall()
        if not rows:
            messagebox.showinfo("Remove habit", "There are no habits to remove.", parent=self.root)
            return
        options = "\n".join(f"{r['id']}: {r['name']} ({r['schedule']})" for r in rows)
        raw = simpledialog.askstring("Remove habit", "Enter the ID of the habit to remove:\n\n" + options,
                                     parent=self.root)
        if raw is None:
            return
        try:
            habit_id = int(raw.strip())
        except ValueError:
            messagebox.showerror("Invalid ID", "Please enter a numeric habit ID.", parent=self.root)
            return
        row = self.db.execute("SELECT name FROM habits WHERE id=?", (habit_id,)).fetchone()
        if not row:
            messagebox.showerror("Not found", "No habit has that ID.", parent=self.root)
            return
        with self.db:
            self.db.execute("DELETE FROM habits WHERE id=?", (habit_id,))
            self.db.execute("DELETE FROM habit_completions WHERE habit_id=?", (habit_id,))
        self.handle_ui_action("remove_habit", {"habit_id": habit_id, "name": row["name"]})
        self.show_habits()

    def show_tasks(self):
        self._clear()
        top = tk.Frame(self.root, bg=BLACK)
        top.pack(fill="x", padx=18)
        self._button(top, "← Home", self.show_home, width=10, font_size=12, accent=False).pack(side="left", pady=12)
        self._title(self.root, "Tasks", "Tap any task to toggle pending / completed")
        controls = tk.Frame(self.root, bg=BLACK)
        controls.pack(pady=(0, 8))
        self._button(controls, "+ Add task", self.add_task, width=13, font_size=12).pack(side="left", padx=5)
        self._button(controls, "Remove completed", self.remove_completed_tasks, width=17, font_size=12, accent=False).pack(side="left", padx=5)

        container = tk.Frame(self.root, bg=BLACK)
        container.pack(fill="both", expand=True, padx=24, pady=6)
        pending = self.db.execute("SELECT * FROM tasks WHERE completed=0 ORDER BY id").fetchall()
        completed = self.db.execute("SELECT * FROM tasks WHERE completed=1 ORDER BY id").fetchall()
        self._task_group(container, "PENDING", pending, False)
        self._task_group(container, "COMPLETED (tap to undo)", completed, True)

    def _task_group(self, parent, heading, rows, completed):
        group = tk.Frame(parent, bg=BLACK, highlightbackground="#454545", highlightthickness=1)
        group.pack(fill="x", pady=7)
        tk.Label(group, text=heading, bg=BLACK, fg=GREEN,
                 font=("Arial", 15, "bold")).pack(anchor="w", padx=12, pady=(8, 4))
        if not rows:
            tk.Label(group, text="Nothing here.", bg=BLACK, fg=MUTED,
                     font=("Arial", 11)).pack(anchor="w", padx=14, pady=7)
        for task in rows:
            prefix = "✓  " if completed else "○  "
            btn = self._button(group, prefix + task["name"],
                               lambda tid=task["id"]: self.toggle_task(tid),
                               width=38, font_size=13, accent=not completed)
            btn.pack(fill="x", padx=12, pady=4)

    def add_task(self):
        name = simpledialog.askstring("Add task", "Enter a name for this task:", parent=self.root)
        if not name or not name.strip():
            return
        with self.db:
            self.db.execute("INSERT INTO tasks(name, completed, created_at) VALUES(?,0,?)",
                            (name.strip(), datetime.now().isoformat(timespec="seconds")))
        self.handle_ui_action("add_task", {"name": name.strip()})
        self.show_tasks()

    def toggle_task(self, task_id):
        row = self.db.execute("SELECT completed FROM tasks WHERE id=?", (task_id,)).fetchone()
        if not row:
            return
        new_value = 0 if row["completed"] else 1
        with self.db:
            self.db.execute("UPDATE tasks SET completed=? WHERE id=?", (new_value, task_id))
        self.handle_ui_action("toggle_task", {"task_id": task_id, "completed": bool(new_value)})
        self.show_tasks()

    def remove_completed_tasks(self):
        count = self.db.execute("SELECT COUNT(*) FROM tasks WHERE completed=1").fetchone()[0]
        if count == 0:
            messagebox.showinfo("Remove completed", "There are no completed tasks to remove.", parent=self.root)
            return
        if messagebox.askyesno("Remove completed", f"Permanently remove {count} completed task(s)?", parent=self.root):
            with self.db:
                self.db.execute("DELETE FROM tasks WHERE completed=1")
            self.handle_ui_action("remove_completed_tasks", {"count": count})
            self.show_tasks()

    def quit_app(self):
        self.handle_ui_action("quit")
        try:
            self.db.close()
        finally:
            self.root.destroy()


def main():
    root = tk.Tk()
    RoutinaApp(root)
    root.mainloop()


if __name__ == "__main__":
    main()


UI ACTION: open_habits | data=None
UI ACTION: quit | data=None
